# Notebook 3 — Despliegue del Modelo (CRISP-DM)
## Plataforma *DatosParaTodos* · datos.gov.co

Este notebook documenta la fase de **Deployment** del proyecto.  
La plataforma tiene **dos formas de despliegue**:

1. **Producción real** — API REST con FastAPI + interfaz VanillaJS (implementada en `app/`)
2. **Prototipo académico** — Interfaz interactiva con **Streamlit** (implementada en este notebook)

Ambas consumen el mismo `Pipeline` sklearn exportado en el Notebook 2.

## 1. Instalación de dependencias

In [ ]:
!pip install streamlit scikit-learn pandas numpy requests --quiet
print('Dependencias instaladas.')

## 2. Verificación del Pipeline exportado

Cargamos el archivo `mejor_modelo_crispdm.pickle` generado en el Notebook 2 y verificamos que el pipeline está completo y funcional.

In [ ]:
import pickle
import numpy as np
import pandas as pd

with open('mejor_modelo_crispdm.pickle', 'rb') as f:
    saved = pickle.load(f)

pipeline   = saved['pipeline']
le         = saved['label_encoder']
num_cols   = saved['num_cols']
cat_cols   = saved['cat_cols']
target_col = saved['target_col']
model_name = saved['model_name']

print('Pipeline cargado exitosamente.')
print(f'Modelo final        : {model_name}')
print(f'Variable objetivo   : {target_col}')
print(f'Clases del modelo   : {le.classes_.tolist()}')
print(f'Variables numéricas : {num_cols}')
print(f'Variables categóric.: {cat_cols}')
print(f'Pasos del pipeline  : {[s[0] for s in pipeline.steps]}')

## 3. Prueba de inferencia con datos reales

Descargamos un nuevo registro de datos.gov.co y lo pasamos por el pipeline para verificar el flujo completo de producción.

In [ ]:
import requests

# Obtener un registro de prueba
URL = 'https://www.datos.gov.co/resource/vjvu-ycr3.json?$limit=5'
test_data = requests.get(URL, timeout=30).json()

if test_data:
    # Tomar el primer registro y remover la columna objetivo
    sample = dict(test_data[0])
    sample.pop(target_col, None)

    df_sample = pd.DataFrame([sample])

    # Asegurar tipos para columnas numéricas
    for col in num_cols:
        if col in df_sample.columns:
            df_sample[col] = pd.to_numeric(df_sample[col], errors='coerce')

    # Predicción
    y_pred  = pipeline.predict(df_sample)
    y_label = le.inverse_transform(y_pred)[0]

    print(f'Clase predicha: {y_label}')

    if hasattr(pipeline, 'predict_proba'):
        proba = pipeline.predict_proba(df_sample)[0]
        print('Probabilidades por clase:')
        for cls, prob in zip(le.classes_, proba):
            print(f'  {cls}: {prob:.2%}')
else:
    print('No se obtuvieron datos de prueba de la API.')

## 4. Despliegue con Streamlit

Escribimos el archivo `app_streamlit.py` y lo ejecutamos con `streamlit run`.  
La interfaz permite al usuario seleccionar un dataset de datos.gov.co, entrenarlo y realizar predicciones interactivas.

In [ ]:
streamlit_code = '''
import pickle
import re
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import requests
import streamlit as st

from sklearn.decomposition import PCA
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier, RandomForestClassifier
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_validate, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier
from statsmodels.stats.multicomp import pairwise_tukeyhsd

st.set_page_config(page_title="DatosParaTodos", page_icon="📊", layout="wide")
st.title("📊 DatosParaTodos — Plataforma de Minería de Datos")
st.caption("Datos abiertos de Colombia · datos.gov.co")

DATE_PATTERN = re.compile(r"\\d{4}-\\d{2}")

DATASETS = {
    "Accidentalidad Vial Bogotá": "https://www.datos.gov.co/resource/vjvu-ycr3.json",
    "Calidad del Aire Bogotá":    "https://www.datos.gov.co/resource/ysvt-9ex4.json",
}

def detect_columns(df):
    sample = df.head(50)
    numeric, date, categorical = [], [], []
    for col in df.columns:
        vals = sample[col].dropna().tolist()
        if not vals:
            categorical.append(col); continue
        num_count  = sum(1 for v in vals if pd.notna(pd.to_numeric(v, errors="coerce")))
        date_count = sum(1 for v in vals if isinstance(v, str) and DATE_PATTERN.search(v))
        if num_count > len(vals) * 0.6: numeric.append(col)
        elif date_count > len(vals) * 0.5: date.append(col)
        else: categorical.append(col)
    return {"numeric": numeric, "date": date, "categorical": categorical}

def clean_data(df):
    df = df.drop_duplicates().reset_index(drop=True)
    col_types = detect_columns(df)
    for col in col_types["numeric"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
            df[col] = df[col].fillna(df[col].median())
    for col in col_types["categorical"]:
        if col in df.columns:
            df[col] = df[col].fillna("No especificado")
    for col in col_types["numeric"]:
        if col not in df.columns: continue
        s = df[col].dropna().sort_values()
        if len(s) < 4 or s.nunique() < 15: continue
        q1, q3 = s.quantile(0.25), s.quantile(0.75)
        iqr = q3 - q1
        if iqr == 0: continue
        df[col] = df[col].clip(q1 - 1.5*iqr, q3 + 1.5*iqr)
    for col in col_types["categorical"]:
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip().str.upper()
    nunique = df.nunique(dropna=True)
    df = df.drop(columns=nunique[nunique <= 1].index.tolist())
    return df, detect_columns(df)

with st.sidebar:
    st.header("⚙️ Configuración")
    dataset_name = st.selectbox("Dataset (datos.gov.co)", list(DATASETS.keys()))
    limit        = st.slider("Registros a descargar", 500, 5000, 2000, step=500)
    run_btn      = st.button("🚀 Ejecutar pipeline CRISP-DM", type="primary")

if run_btn:
    url = DATASETS[dataset_name] + f"?$limit={limit}"
    with st.spinner("Descargando datos..."):
        try:
            raw = requests.get(url, timeout=30).json()
            df_raw = pd.DataFrame(raw)
        except Exception as e:
            st.error(f"Error al descargar datos: {e}"); st.stop()

    st.success(f"✅ Datos descargados: {len(df_raw)} registros × {df_raw.shape[1]} variables")

    with st.spinner("Aplicando pipeline de limpieza..."):
        df, col_types = clean_data(df_raw)

    with st.expander("🔍 Vista previa del dataset limpio"):
        st.dataframe(df.head(10), use_container_width=True)

    cat_cols_all = [c for c in df.select_dtypes(exclude=[np.number]).columns]
    target_col = None
    for col in cat_cols_all:
        if 2 <= df[col].nunique() <= 20:
            target_col = col; break

    if not target_col:
        st.error("No se detectó variable objetivo."); st.stop()

    st.info(f"🎯 Variable objetivo: **{target_col}** ({df[target_col].nunique()} clases)")

    df = df.dropna(subset=[target_col])
    y_raw = df[target_col].astype(str)
    X_df  = df.drop(columns=[target_col])
    num_cols = X_df.select_dtypes(include=[np.number]).columns.tolist()
    cat_cols = X_df.select_dtypes(exclude=[np.number]).columns.tolist()

    if len(num_cols) > 1:
        corr = X_df[num_cols].corr().abs()
        upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
        redundant = [c for c in upper.columns if any(upper[c] > 0.90)]
        if redundant:
            X_df = X_df.drop(columns=redundant)
            num_cols = [c for c in num_cols if c not in redundant]

    preprocessor = ColumnTransformer([
        ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                          ("sc",  StandardScaler()),
                          ("pca", PCA(n_components=0.95, random_state=42))]), num_cols),
        ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                          ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), cat_cols)
    ], remainder="drop")

    le = LabelEncoder()
    y  = le.fit_transform(y_raw)
    X_processed = preprocessor.fit_transform(X_df)

    X_train, X_test, y_train, y_test = train_test_split(
        X_processed, y, test_size=0.30, random_state=42, stratify=y)

    counts = np.bincount(y_train)
    ratio  = counts.min() / counts.max() if counts.max() > 0 else 1.0
    if ratio < 0.80:
        try:
            from imblearn.over_sampling import SMOTE
            k = min(5, max(1, counts.min() - 1))
            X_train, y_train = SMOTE(random_state=42, k_neighbors=k).fit_resample(X_train, y_train)
            st.info("⚖️ SMOTE aplicado al 70% de entrenamiento.")
        except: pass

    n_classes = len(np.unique(y))
    avg = "weighted" if n_classes > 2 else "binary"

    MODELS = {
        "Regresión Logística":          LogisticRegression(max_iter=1000, random_state=42),
        "Árbol de Decisión":            DecisionTreeClassifier(random_state=42),
        "K-Vecinos (KNN)":              KNeighborsClassifier(n_neighbors=min(5, len(X_train)-1)),
        "Red Neuronal (MLP)":           MLPClassifier(hidden_layer_sizes=(50,), max_iter=500, random_state=42),
        "Random Forest (Ensamble)":     RandomForestClassifier(n_estimators=100, random_state=42),
        "Gradient Boosting (Ensamble)": GradientBoostingClassifier(n_estimators=100, random_state=42),
        "AdaBoost (Ensamble)":          AdaBoostClassifier(n_estimators=100, random_state=42),
    }

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    results = []
    progress = st.progress(0)
    for idx, (name, model) in enumerate(MODELS.items()):
        try:
            cv_res = cross_validate(model, X_train, y_train, cv=cv, scoring=["accuracy"])
            acc_scores = cv_res["test_accuracy"]
            model.fit(X_train, y_train)
            y_pred = model.predict(X_test)
            results.append({
                "Modelo": name, "CV Acc": round(float(acc_scores.mean()), 4),
                "CV Scores": acc_scores.tolist(), "_model": model,
                "Accuracy":  round(accuracy_score(y_test, y_pred), 4),
                "F1":        round(f1_score(y_test, y_pred, average=avg, zero_division=0), 4),
            })
        except: pass
        progress.progress((idx + 1) / len(MODELS))

    df_res = pd.DataFrame(results).drop(columns=["CV Scores", "_model"])
    st.subheader("📈 Resultados de los 7 modelos")
    st.dataframe(df_res.sort_values("CV Acc", ascending=False).reset_index(drop=True),
                 use_container_width=True)

    # ANOVA + Tukey HSD
    from scipy import stats as sp_stats
    grupos  = [r["CV Scores"] for r in results]
    nombres = [r["Modelo"]    for r in results]
    f_stat, p_val = sp_stats.f_oneway(*grupos)
    significativo = p_val < 0.05

    col1, col2 = st.columns(2)
    col1.metric("ANOVA F-statistic", f"{f_stat:.4f}")
    col2.metric("ANOVA p-valor", f"{p_val:.6f}",
                delta="Significativo ✓" if significativo else "No significativo")

    all_sc = np.concatenate([np.array(g) for g in grupos])
    all_lb = np.concatenate([[n] * len(g) for n, g in zip(nombres, grupos)])
    tukey  = pairwise_tukeyhsd(endog=all_sc, groups=all_lb, alpha=0.05)
    tukey_rows = [
        {"A": str(r[0]), "B": str(r[1]), "Dif.": round(float(r[2]),4),
         "p-adj": round(float(r[3]),6), "Sig.": "✓" if bool(r[6]) else ""}
        for r in tukey.summary().data[1:]
    ]
    with st.expander("🔬 Tukey HSD — comparaciones por pares"):
        st.dataframe(pd.DataFrame(tukey_rows), use_container_width=True)

    WEIGHTS = {
        "Regresión Logística": 1, "Árbol de Decisión": 2, "K-Vecinos (KNN)": 3,
        "Red Neuronal (MLP)": 4, "AdaBoost (Ensamble)": 5,
        "Gradient Boosting (Ensamble)": 6, "Random Forest (Ensamble)": 7
    }
    if significativo:
        top3 = sorted(results, key=lambda r: r["CV Acc"], reverse=True)[:3]
    else:
        top3 = sorted(results, key=lambda r: (WEIGHTS.get(r["Modelo"], 99), -r["CV Acc"]))[:3]

    top3_names = [r["Modelo"] for r in top3]
    st.success(f"🏆 TOP 3 seleccionados: {', '.join(top3_names)}")

    # GridSearchCV sobre TOP 3
    st.subheader("🔎 GridSearchCV — TOP 3 modelos")
    GS_PARAMS = {
        "Regresión Logística":          {"C": [0.01, 0.1, 1.0, 10.0, 100.0]},
        "Árbol de Decisión":            {"max_depth": [3, 5, 10, 15], "min_samples_split": [2, 5, 10]},
        "K-Vecinos (KNN)":              {"n_neighbors": [3, 5, 7, 11], "weights": ["uniform", "distance"]},
        "Red Neuronal (MLP)":           {"alpha": [1e-4, 1e-3, 1e-2], "hidden_layer_sizes": [(50,), (100,)]},
        "Random Forest (Ensamble)":     {"n_estimators": [50, 100, 200], "max_depth": [5, 10, None]},
        "Gradient Boosting (Ensamble)": {"n_estimators": [50, 100, 200], "learning_rate": [0.05, 0.1, 0.2]},
        "AdaBoost (Ensamble)":          {"n_estimators": [50, 100, 200], "learning_rate": [0.5, 1.0]},
    }
    gs_rows = []
    with st.spinner("Ejecutando GridSearch..."):
        for r in top3:
            name = r["Modelo"]
            params = GS_PARAMS.get(name, {})
            if not params: continue
            try:
                gs = GridSearchCV(r["_model"].__class__(**r["_model"].get_params()),
                                  params, cv=3, scoring="accuracy", n_jobs=-1, refit=True)
                gs.fit(X_train, y_train)
                test_a = accuracy_score(y_test, gs.best_estimator_.predict(X_test))
                gs_rows.append({"Modelo": name, "Params": str(gs.best_params_),
                                 "CV GS": round(gs.best_score_, 4),
                                 "Test": round(test_a, 4)})
            except Exception as e:
                gs_rows.append({"Modelo": name, "Params": f"Error: {e}",
                                 "CV GS": 0, "Test": 0})
    st.dataframe(pd.DataFrame(gs_rows), use_container_width=True)

    # Optimización Bayesiana (Optuna)
    st.subheader("🧠 Optimización Bayesiana (Optuna TPE) — TOP 3 modelos")
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)

    def build_model(name, params):
        if name == "Regresión Logística":
            return LogisticRegression(C=params.get("C", 1.0), max_iter=1000, random_state=42)
        elif name == "Árbol de Decisión":
            return DecisionTreeClassifier(max_depth=params.get("max_depth", 5), random_state=42)
        elif name == "K-Vecinos (KNN)":
            return KNeighborsClassifier(n_neighbors=params.get("n_neighbors", 5))
        elif name == "Red Neuronal (MLP)":
            return MLPClassifier(alpha=params.get("alpha", 1e-4),
                                 hidden_layer_sizes=params.get("hidden_layer_sizes", (50,)),
                                 max_iter=500, random_state=42)
        elif name == "Random Forest (Ensamble)":
            return RandomForestClassifier(n_estimators=params.get("n_estimators", 100),
                                          max_depth=params.get("max_depth", 10), random_state=42)
        elif name == "Gradient Boosting (Ensamble)":
            return GradientBoostingClassifier(n_estimators=params.get("n_estimators", 100),
                                              learning_rate=params.get("learning_rate", 0.1), random_state=42)
        elif name == "AdaBoost (Ensamble)":
            return AdaBoostClassifier(n_estimators=params.get("n_estimators", 100),
                                      learning_rate=params.get("learning_rate", 0.1), random_state=42)

    best_score, best_weight, best_name, best_model = -1, 99, "", None
    opt_rows = []

    with st.spinner("Optimización bayesiana en TOP 3 modelos..."):
        for r in top3:
            name = r["Modelo"]
            def objective(trial, _name=name):
                if _name == "Regresión Logística": p = {"C": trial.suggest_float("C", 1e-4, 1e2, log=True)}
                elif _name == "Árbol de Decisión": p = {"max_depth": trial.suggest_int("max_depth", 2, 20)}
                elif _name == "K-Vecinos (KNN)": p = {"n_neighbors": trial.suggest_int("n_neighbors", 1, min(15, len(X_train)-1))}
                elif _name == "Red Neuronal (MLP)": p = {"alpha": trial.suggest_float("alpha", 1e-5, 1e-1, log=True),
                    "hidden_layer_sizes": trial.suggest_categorical("hidden_layer_sizes", [(50,), (50,50), (100,)])}
                elif _name == "Random Forest (Ensamble)": p = {"n_estimators": trial.suggest_int("n_estimators", 10, 200),
                    "max_depth": trial.suggest_int("max_depth", 2, 20)}
                elif _name == "Gradient Boosting (Ensamble)": p = {"n_estimators": trial.suggest_int("n_estimators", 10, 200),
                    "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True)}
                elif _name == "AdaBoost (Ensamble)": p = {"n_estimators": trial.suggest_int("n_estimators", 10, 200),
                    "learning_rate": trial.suggest_float("learning_rate", 0.01, 1.0, log=True)}
                else: raise optuna.exceptions.TrialPruned()
                m = build_model(_name, p)
                return cross_validate(m, X_train, y_train, cv=3, scoring="accuracy", n_jobs=-1)["test_score"].mean()

            study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
            study.optimize(objective, n_trials=15, timeout=90)
            opt_model = build_model(name, study.best_params)
            opt_model.fit(X_train, y_train)
            opt_score = study.best_value
            test_a    = accuracy_score(y_test, opt_model.predict(X_test))
            opt_rows.append({"Modelo": name, "Params": str(study.best_params),
                              "CV Optuna": round(opt_score, 4), "Test": round(test_a, 4)})

            weight    = WEIGHTS.get(name, 99)
            is_better = (weight < best_weight) if not significativo else (opt_score > best_score)
            if is_better or (weight == best_weight and opt_score > best_score):
                best_score, best_weight, best_name, best_model = opt_score, weight, name, opt_model

    st.dataframe(pd.DataFrame(opt_rows), use_container_width=True)
    st.success(f"🥇 Mejor modelo: **{best_name}** (CV Optuna: {best_score:.4f})")

    # Pipeline final
    final_pipeline = Pipeline([("preprocessor", preprocessor), ("classifier", best_model)])
    final_pipeline.fit(X_df, y)

    import io
    buf = io.BytesIO()
    pickle.dump({"pipeline": final_pipeline, "label_encoder": le,
                 "num_cols": num_cols, "cat_cols": cat_cols,
                 "target_col": target_col, "model_name": best_name}, buf)
    buf.seek(0)
    st.download_button("⬇️ Descargar Pipeline (.pickle)", data=buf,
                       file_name="mejor_modelo_crispdm.pickle", mime="application/octet-stream")

    st.session_state["pipeline"] = final_pipeline
    st.session_state["le"]       = le
    st.session_state["num_cols"] = num_cols
    st.session_state["cat_cols"] = cat_cols
    st.session_state["target"]   = target_col

if "pipeline" in st.session_state:
    st.divider()
    st.subheader("🔮 Predicción interactiva")
    pipeline_loaded = st.session_state["pipeline"]
    le_loaded       = st.session_state["le"]
    num_cols_loaded = st.session_state["num_cols"]
    cat_cols_loaded = st.session_state["cat_cols"]
    target_loaded   = st.session_state["target"]

    st.write(f"Variable a predecir: **{target_loaded}** | Clases: {le_loaded.classes_.tolist()}")
    input_data = {}
    all_input_cols = num_cols_loaded + cat_cols_loaded
    cols_form  = st.columns(min(3, max(1, len(all_input_cols))))
    for i, col in enumerate(all_input_cols):
        c = cols_form[i % len(cols_form)]
        if col in num_cols_loaded:
            input_data[col] = c.number_input(col, value=0.0, key=f"inp_{col}")
        else:
            input_data[col] = c.text_input(col, value="", key=f"inp_{col}")

    if st.button("🎯 Predecir", type="primary"):
        try:
            df_input = pd.DataFrame([input_data])
            for col in num_cols_loaded:
                if col in df_input.columns:
                    df_input[col] = pd.to_numeric(df_input[col], errors="coerce")
            y_pred  = pipeline_loaded.predict(df_input)
            y_label = le_loaded.inverse_transform(y_pred)[0]
            st.success(f"✅ Resultado predicho: **{y_label}**")
            if hasattr(pipeline_loaded, "predict_proba"):
                proba    = pipeline_loaded.predict_proba(df_input)[0]
                df_proba = pd.DataFrame({"Clase": le_loaded.classes_,
                                          "Probabilidad": [f"{p:.2%}" for p in proba]})
                st.dataframe(df_proba, use_container_width=True)
        except Exception as e:
            st.error(f"Error en predicción: {e}")
'''

with open('app_streamlit.py', 'w', encoding='utf-8') as f:
    f.write(streamlit_code.strip())

print('app_streamlit.py generado exitosamente (con Tukey HSD + GridSearch + Optuna).')
print('Para ejecutar: streamlit run app_streamlit.py')

## 5. Lanzar Streamlit desde el notebook

> **Instrucción:** Abre una terminal y ejecuta el comando de abajo. La interfaz se abre automáticamente en el navegador en `http://localhost:8501`.

In [ ]:
# Ejecutar en una terminal separada:
# streamlit run app_streamlit.py

# O desde aquí en modo background (Jupyter):
import subprocess, sys
proc = subprocess.Popen([sys.executable, '-m', 'streamlit', 'run', 'app_streamlit.py',
                          '--server.headless', 'true'])
print(f'Streamlit iniciado (PID {proc.pid})')
print('Abre http://localhost:8501 en tu navegador.')

## 6. Arquitectura de la plataforma de producción

La solución completa **DatosParaTodos** está desplegada con la siguiente arquitectura:

```
┌─────────────────────────────────────────────────────┐
│                  FRONTEND (VanillaJS)                │
│  Presentation/ → index.html + secciones temáticas   │
│  Chat IA · Analytics · Modeling · Visualizaciones   │
└────────────────────┬────────────────────────────────┘
                     │ HTTP REST
┌────────────────────▼────────────────────────────────┐
│              BACKEND (FastAPI + Python)              │
│  app/routers/analytics.py  → Limpieza de datos      │
│  app/routers/modeling.py   → Pipeline CRISP-DM      │
│  app/routers/chat.py       → Chat IA (Gemini)       │
│  app/services/modeling.py  → Motor ML completo      │
│  app/services/analytics.py → Motor de limpieza      │
└────────────────────┬────────────────────────────────┘
                     │
┌────────────────────▼────────────────────────────────┐
│              DATOS (datos.gov.co)                   │
│  API SODA pública · +40 datasets temáticos          │
│  Economía · Salud · Educación · Movilidad · Medio   │
│  ambiente · Seguridad · Servicios · Espacio         │
└─────────────────────────────────────────────────────┘
```

**Flujo completo CRISP-DM en producción:**
1. El usuario selecciona un dataset de datos.gov.co
2. El backend descarga los datos vía API SODA
3. Se ejecuta el pipeline de limpieza (`AnalyticsService`): deduplicación → imputación → IQR outliers → normalización
4. Se ejecuta el pipeline de modelado (`ModelingService`): varianza cero → correlaciones → PCA → 7 modelos → CV → ANOVA/Bonferroni → Optuna TPE → Pipeline final
5. El modelo empaquetado queda disponible para predicciones interactivas en tiempo real